# Arm **SV3** — SA²-v2 with a zero-init gate on x^AS (`c̃ = c + tanh(g)·x^AS`), seed 2024

Rule (written before any number): `core/docs/final/SA2V2_RUN.md` §10. Upload branch `v_final_stp_v2` to
`Thesis-Final/kat-vad`. Needs what `sa2v2_nexar_dada.ipynb` and `sv2_w0d_probe.ipynb` left on Drive: the Nexar /
DADA / DoTA-CAP-dev cell caches, and the W0, W0D, SV2 reads.

| step | what | GPU | estimate |
|---|---|---|---|
| 1–3 | setup, preflight | — | 5–10 min |
| 4 | pytest on the new code — **stop if red** | no | 2–5 min |
| 5 | train SV3 (1,749 steps; gate in its own AdamW group at 25 × lr) | **yes** | 15–45 min |
| 6 | SV3: DADA read + DoTA-CAP-dev read (cells from the SV2 run) | **yes** | 10–20 min |
| 7 | read-out + probe `adam` on SV3 → **paste `REPORTS/sv3_readout.md` and `REPORTS/probe_sv3/sv2_probe_adam.md`** | no | 5 min |

Estimates only. Every step skips what is already on Drive. One seed ⇒ **PRELIMINARY (n = 1 seed)**.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_window, A2 bakes, KNN, v2 A2 runs
DRIVE_V5 = '/content/drive/MyDrive/Thesis-V5'     # read-only: 14 x 14 CLIP token grids (branch v5-r7)
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code + every cache / output this notebook writes
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_stp_v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'SV3_ARM'), 'this upload predates SV3 -- re-upload kat-vad/'
assert '## 10. Arm SV3' in (REPO / 'core/docs/final/SA2V2_RUN.md').read_text(), 'SA2V2_RUN.md has no §10 (SV3 rule) in this upload'

SEED, ARM, SHUF = constants.SV2_SEED, constants.SV2_ARM, constants.SV2_SHUFFLED_ARM
W0D, SV3 = constants.W0D_ARM, constants.SV3_ARM
ENC      = constants.V5_ENCODER
C        = constants.NEXAR_CONSTRUCTION_WINDOW
DATASET  = constants.NEXAR_DATASET
T2NAME   = constants.DADA_ORIGIN_DATASET
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
S_N, S_DADA, S_DOTA = constants.STP_NEXAR_STRIDE, constants.ZS_STRIDE, constants.STP_DOTA_STRIDE

V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'
BUILD    = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n4' / 'build_report.json'
DATA     = Path(DRIVE_V2) / 'data' / f'{DATASET}_{C}'
A2_NEXAR = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'
A2_DADA  = V2_CACHE / 'v2' / 'A2_S' / f'{T2NAME}_test_sources_nexar'
A2_DOTA  = V2_CACHE / 'v2' / 'A2_S' / f'{CAP_DEV}_s3_from_s1_nexar'
KNN      = V2_CACHE / 'knn' / f'{DATASET}_{C}' / constants.KNN_CACHE_FILENAME
A2_RUNS  = V2_OUT / constants.NEXAR_FULL_RUNS_DIR / 'A2'
GRIDS    = Path(DRIVE_V5) / 'cache' / 'video_grid' / 'clip_vit_b16'
GRID_N   = GRIDS / f'{DATASET}_s{S_N}_cgrid14'
GRID_D   = GRIDS / f'{constants.DOTA_CAP_DATASET}_s{S_DOTA}_cgrid14'
T2       = constants.DATA_ROOT / T2NAME
T2_CLIP  = Path(DRIVE) / 'cache' / 'clip' / T2NAME
T2_CENSUS = T2 / 'counts' / 'resolved_census.json'
ANNOT    = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME / constants.DADA_ORIGIN_ANNOTATION_FILENAME
ARC_DADA = Path(DRIVE_F) / 'data' / constants.ZS_FRAMES_DIRNAME / T2NAME
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
ZS_ROOT  = constants.OUTPUT_ROOT / constants.ZS_OUTPUT_DIRNAME
W0_RUN   = constants.OUTPUT_ROOT / constants.WU_OUTPUT_DIRNAME / constants.ZS_CONTROL_ARM / f's{SEED}'
W0_DADA  = ZS_ROOT / 'eval' / 'dada' / constants.ZS_CONTROL_ARM
R4_DADA  = ZS_ROOT / 'eval' / 'dada' / constants.ZS_SA2_ARM
W0_DOTA  = constants.OUTPUT_ROOT / constants.WU_OUTPUT_DIRNAME / 'eval' / 'dota' / CAP_DEV / constants.ZS_CONTROL_ARM

from core.data import stp_patches as stp_data      # noqa: E402

CELL_N, CELL_DADA = stp_data.cell_cache_dir(DATASET, S_N), stp_data.cell_cache_dir(T2NAME, S_DADA)
CELL_DOTA = stp_data.cell_cache_dir(constants.DOTA_CAP_DATASET, S_DOTA)
ROOT     = constants.OUTPUT_ROOT / constants.SV2_OUTPUT_DIRNAME
REPORTS  = ROOT / 'REPORTS'
RUN_DIR  = {ARM: ROOT / ARM / f's{SEED}', SHUF: ROOT / SHUF / f's{SEED}', W0D: ROOT / W0D / f's{SEED}', SV3: ROOT / SV3 / f's{SEED}'}
PROBE    = REPORTS / constants.SV2_PROBE_DIRNAME
LOCAL    = Path('/content/sv2')
for p in (REPORTS, LOCAL):
    p.mkdir(parents=True, exist_ok=True)

PLAN = json.loads(BUILD.read_text())['plan'][C]
STEPS, EPOCHS = PLAN['steps'], PLAN['epochs']     # 1,749 = 33 x 53
for name, path in (('Nexar_window', DATA), ('Nexar A2 bake', A2_NEXAR), ('DADA A2 bake', A2_DADA), ('KNN', KNN),
                   ('v2 A2 runs', A2_RUNS), ('Nexar tokens', GRID_N), ('T2 meta', T2 / constants.META_FILENAME),
                   ('T2 CLIP', T2_CLIP), ('T2 census', T2_CENSUS), ('DADA annotation', ANNOT),
                   ('DADA frames', ARC_DADA / constants.ZS_FRAMES_MANIFEST), ('W0 run', W0_RUN / 'metrics.jsonl'),
                   ('W0 DADA read', W0_DADA / 'clip_aucs.json')):
    assert path.exists(), f'missing {name}: {path}'
    print(f'  {name:16s} {path}')
print(f'steps {STEPS} ({EPOCHS} epochs) | cells -> {CELL_N} | {CELL_DADA}\nruns/reads -> {ROOT}')
for name, path in (('SV2 DADA read', ROOT / 'eval' / 'dada' / ARM / 'clip_aucs.json'),
                   ('SV2 DoTA read', ROOT / 'eval' / 'dota' / ARM / 'clip_aucs.json'),
                   ('W0D DADA read', ROOT / 'eval' / 'dada' / W0D / 'clip_aucs.json'),
                   ('W0D DoTA read', ROOT / 'eval' / 'dota' / W0D / 'clip_aucs.json'),
                   ('W0 DoTA read', W0_DOTA / 'clip_aucs.json'), ('Nexar cells', CELL_N / constants.SV2_LEVEL_FILENAME),
                   ('DADA cells', CELL_DADA), ('DoTA cells', CELL_DOTA)):
    assert path.exists(), f'missing {name}: {path} -- run sa2v2_nexar_dada.ipynb (parts A, C) and sv2_w0d_probe.ipynb first'
print('W0 / W0D / SV2 reads and the three cell caches found')


In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl xlrd
python -c "import av, einops, faiss, sklearn, openpyxl; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 5-7, 9, 10 need one'
df -h /content | tail -1

In [ ]:
import pickle
import shutil
import subprocess
import threading
import time

import numpy as np
import torch
import yaml

from core.tools.dada_whole_eval import test_sources    # noqa: E402
from core.tools.sa2v2_cells import training_sources    # noqa: E402
from core.tools.v5_stage1 import _flat                 # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF (absl eats INFO logs)


def run(cmd, log=None):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


def stage(src, dst, ids=None):
    """Copy a Drive dir (only ``ids`` + json manifests if given) or file to VM disk once (bit-identical; C2)."""
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    if ids is None:
        for f in src.iterdir():
            if f.is_file() and not (dst / f.name).exists():
                shutil.copy2(f, dst / f.name)
    else:
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    """VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10)."""
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    t0 = time.time()
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    print(f'Drive flushed and remounted in {time.time() - t0:.0f}s')


def rows(d, v):
    return int(np.load(d / f'{v}.npy', mmap_mode='r').shape[0])


def ckpt_step(path):
    """The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step)."""
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def is_verified(drive_dir):
    rec, ckpt = drive_dir / 'verified.json', drive_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == STEPS and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


class GpuPeak:
    """Peak GPU memory of a block, sampled with nvidia-smi (proposal analysis 6: cost)."""

    def __enter__(self):
        self.peak, self.stop = 0, threading.Event()
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()
        return self

    def _loop(self):
        while not self.stop.wait(5):
            out = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                                 capture_output=True, text=True).stdout.split()
            if out:
                self.peak = max(self.peak, int(out[0]))

    def __exit__(self, *exc):
        self.stop.set()
        self.thread.join()
        print(f'peak GPU memory: {self.peak} MiB')


# ids and the preflight: nothing below runs on a wrong corpus
NEXAR_IDS = training_sources(DATA)
DADA_IDS = test_sources(T2)
assert len(NEXAR_IDS) == 451, f'{len(NEXAR_IDS)} Nexar training sources, expected 451'
assert len(DADA_IDS) == 370, len(DADA_IDS)
miss = [v for v in NEXAR_IDS if not (GRID_N / f'{v}.npy').exists()]
assert not miss, f'{len(miss)} Nexar sources have no token grid in {GRID_N}: {miss[:5]}'
miss = [v for v in DADA_IDS if not (A2_DADA / f'{v}.npy').exists()]
assert not miss, f'{len(miss)} DADA sources have no A2 bake: {miss[:5]}'
g = np.load(GRID_N / f'{NEXAR_IDS[0]}.npy', mmap_mode='r')
assert g.size == g.shape[0] * 14 * 14 * 512, f'token grid has shape {g.shape}, expected (L, 14, 14, 512)'
man = json.loads((ARC_DADA / constants.ZS_FRAMES_MANIFEST).read_text())
assert man['pass'] and man['stride'] == S_DADA, man
for name, d in (('Nexar', A2_NEXAR), ('DADA', A2_DADA)):
    m = json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
    assert (m['crn'], m['motion']) == (constants.V2_OFF, ENC), f'{name} bake is {m}'
same = {json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())['train_ids_sha1'] for d in (A2_NEXAR, A2_DADA)}
assert len(same) == 1, f'the bakes carry different statistics: {same}'
ID_N, ID_DADA = write_ids(LOCAL / 'nexar.txt', NEXAR_IDS), write_ids(LOCAL / 'dada.txt', DADA_IDS)
L_DEFS = LOCAL / 'nexar_defs'
L_DEFS.mkdir(parents=True, exist_ok=True)
for f in DATA.glob('*.json'):               # defs.json etc.: the definitions the arms trained with
    shutil.copy2(f, L_DEFS / f.name)
print(f'preflight OK: {len(NEXAR_IDS)} Nexar sources with tokens, {len(DADA_IDS)} DADA sources, shared statistics')

## 4. Tests on the new code (CPU) — **stop and paste the output if anything fails**


In [ ]:
%%bash
cd "$REPO"
python -m pytest -q -p no:cacheprovider core/tests/test_sv3.py core/tests/test_w0d_probe.py core/tests/test_sa2v2.py core/tests/test_wu_arms.py 2>&1 | tail -15


## 5. Train SV3 (GPU)

SV2's recipe + `stp.sv2_gate=true`. **Watch the log: past 2 s/step, stop and paste it to Claude.**
Then J10, the config diff (vs v2 A2 and vs SV2: only the gate keys), the gate's own AdamW group.


In [ ]:
def train_cmd(out_dir, data, knn):
    cmd = [sys.executable, '-m', 'core.train',
           '--set', 'train.stage=2', '--set', f'train.seed={SEED}',
           '--set', f'train.num_epochs={EPOCHS}', '--set', 'train.amp=true',
           '--set', f'data.dataset={DATASET}',
           '--set', f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}',
           '--set', f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}',
           '--set', 'kip.enabled=false', '--set', 'loss.dvs_anchor_mode=span',
           '--set', 'data.is_egocentric=false',
           '--set', f'v2.motion={ENC}', '--set', f'model.motion_dim={constants.V5_GRID_DIM}',
           '--set', 'stp.sa2v2=true', '--set', 'stp.sv2_gate=true',
           '--data-dir', data, '--clip-dir', stage(A2_NEXAR, LOCAL / 'stage' / 'a2_nexar', NEXAR_IDS),
           '--patch-dir', stage(CELL_N, LOCAL / 'stage' / 'cell_nexar', NEXAR_IDS), '--output-dir', out_dir]
    return cmd + (['--knn-cache', knn] if knn else [])


def train_arm(arm, data, knn):
    drive_dir, local = RUN_DIR[arm], LOCAL / 'runs' / arm
    if is_verified(drive_dir):
        print(f'{arm} s{SEED}: verified on Drive -- not retrained')
        return
    assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    t0 = time.time()
    with GpuPeak():
        run(train_cmd(local, data, knn) + resume, log=LOCAL / f'train_{arm}.log')
    last = metrics_last(local / 'metrics.jsonl')
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{arm}: J10 locally'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        publish(local / name, drive_dir / name)
    flush_drive()
    assert ckpt_step(drive_dir / 'checkpoint_last.pt') == STEPS, f'{arm}: Drive copy is not step {STEPS}'
    (LOCAL / 'verified.json').write_text(json.dumps(
        {'step': STEPS, 'ckpt_bytes': (drive_dir / 'checkpoint_last.pt').stat().st_size}))
    publish(LOCAL / 'verified.json', drive_dir / 'verified.json')
    print(f'{arm}: {(time.time() - t0) / 60:.0f} min | entropy {last.get("sv2_entropy")} | gamma {last.get("sv2_gamma")} '
          f'| beta1 {last.get("sv2_beta1")} | x^AS / c {last.get("sv2_xas_ratio")} | tanh(g) {last.get("sv2_gate")}')


def check_run(arm):
    d = RUN_DIR[arm]
    payload = torch.load(d / 'checkpoint_last.pt', map_location='cpu', weights_only=False)
    assert payload['global_step'] == metrics_last(d / 'metrics.jsonl')['global_step'] == STEPS, f'{arm}: J10'
    state = payload['model']
    level = json.loads((CELL_N / constants.SV2_LEVEL_FILENAME).read_text())
    want = [level['mo_mean'], level['mo_std'], level['rho_mean'], level['rho_std']]
    assert np.allclose(state['sa2v2.level'].numpy(), want, rtol=1e-5), f'{arm}: level constants {state["sa2v2.level"]}'
    assert not torch.equal(state['sa2v2.text_projection'], torch.eye(512)), f'{arm}: tau_delta has no CLIP projection'
    ref = _flat(yaml.safe_load((A2_RUNS / f's{SEED}' / 'config.yaml').read_text()))
    got = _flat(yaml.safe_load((d / 'config.yaml').read_text()))
    diff = {k: (ref.get(k), got.get(k)) for k in set(ref) | set(got)
            if not k.startswith(('stp.', 'v5.')) and k not in ('model.motion_init_std', 'train.motion_lr_mult')
            and ref.get(k) != got.get(k) and not (k == 'model.motion_dropout' and not got.get(k))}
    assert not diff, f'{arm} differs from v2 A2 s{SEED} beyond stp / v5 / W_u knobs: {diff}'
    assert got['stp.sa2v2'] is True and got['stp.sa2'] is False, arm
    assert got['stp.text_prompt'] == constants.STP_PROMPT_RELEASED, f'{arm}: prefix is not OFF'
    assert got['stp.sv2_gate'] is True and got['stp.gate_lr_mult'] == constants.SV3_GATE_LR_MULT, f'{arm}: gate off'
    assert 'sa2v2.gate' in state and len(payload['optimizer']['param_groups']) == 2, f'{arm}: no gate group'
    ref_sv2 = _flat(yaml.safe_load((RUN_DIR[ARM] / 'config.yaml').read_text()))
    d_sv2 = {k for k in set(ref_sv2) | set(got) if ref_sv2.get(k) != got.get(k)
             and not (k == 'model.motion_dropout' and not got.get(k))}
    assert d_sv2 == {'stp.sv2_gate', 'stp.gate_lr_mult'} or d_sv2 == {'stp.sv2_gate'}, f'{arm} vs SV2: {d_sv2}'
    print(f'{arm}: J10 {STEPS}, level + projection stored, config == v2 A2 s{SEED} + stp.sa2v2, prefix OFF')


data_local = stage(DATA, LOCAL / 'stage' / 'data')
knn_local = stage(KNN, LOCAL / 'stage' / 'knn.npz')
train_arm(SV3, data_local, knn_local)
check_run(SV3)

## 6. SV3 reads (GPU): DADA-2000 whole video + DoTA-CAP-dev protocol B, SV2's protocols


In [ ]:
from core.data.v2_splits import load_split    # noqa: E402

assert torch.cuda.is_available(), 'torch sees no GPU'
CKPT3 = RUN_DIR[SV3] / 'checkpoint_last.pt'
L_T2 = LOCAL / 'T2_meta'
L_T2.mkdir(parents=True, exist_ok=True)
shutil.copy2(T2 / constants.META_FILENAME, L_T2 / constants.META_FILENAME)
DADA_OUT, DOTA_OUT = ROOT / 'eval' / 'dada', ROOT / 'eval' / 'dota'
if not (DADA_OUT / SV3 / 'clip_aucs.json').exists():         # SV2's protocol: A2 bake + DADA cells
    local = LOCAL / 'dada' / SV3
    run([sys.executable, '-m', 'core.tools.dada_whole_eval', '--run', f's{SEED}', CKPT3, '--t2-dir', L_T2,
         '--annotation', ANNOT, '--census', T2_CENSUS, '--input-dir', stage(A2_DADA, LOCAL / 'bake_dada', DADA_IDS),
         '--patch-dir', stage(CELL_DADA, LOCAL / 'cell_dada', DADA_IDS), '--data-dir', L_DEFS, '--out-dir', local],
        log=LOCAL / 'dada_sv3.log')
    publish(local, DADA_OUT / SV3)
r = json.loads((DADA_OUT / SV3 / 'dada_readout.json').read_text())
print(f"dada {SV3}: macro {r['macro']['mean']:.4f} over {r['two_class']} sources | micro {r['micro']:.4f}")

if not (DOTA_OUT / SV3 / 'clip_aucs.json').exists():         # protocol B + DoTA-CAP-dev cells
    CAP_IDS = sorted(load_split(CAP_DEV))
    local = LOCAL / 'dota' / SV3
    run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', CKPT3,
         '--input-dir', stage(A2_DOTA, LOCAL / 'bake_dota', CAP_IDS), '--patch-dir', stage(CELL_DOTA, LOCAL / 'cell_dota', CAP_IDS),
         '--s1-dir', stage(DOTA_S1, LOCAL / 'dota_s1', CAP_IDS), '--metadata', DOTA / 'metadata_val.json',
         '--split-file', DOTA / 'val_split.txt', '--data-dir', stage(DOTA / 'labels_s8', LOCAL / 'dota_labels'),
         '--split', CAP_DEV, '--out-dir', local, '--device', 'auto'], log=LOCAL / 'dota_sv3.log')
    publish(local, DOTA_OUT / SV3)
r = json.loads((DOTA_OUT / SV3 / 'protocol_b_readout.json').read_text())
print(f"dota {SV3}: macro {r['macro']['mean']:.4f}")
flush_drive()


## 7. Read-out + probe `adam` on SV3 → **paste both `.md` to Claude**


In [ ]:
E = ROOT / 'eval'
run([sys.executable, '-m', 'core.tools.sv3_readout', '--sv3-run', RUN_DIR[SV3],
     '--w0-dada', W0_DADA, '--w0d-dada', E / 'dada' / W0D, '--sv2-dada', E / 'dada' / ARM, '--sv3-dada', E / 'dada' / SV3,
     '--w0-dota', W0_DOTA, '--w0d-dota', E / 'dota' / W0D, '--sv2-dota', E / 'dota' / ARM, '--sv3-dota', E / 'dota' / SV3,
     '--out-dir', REPORTS])
print((REPORTS / 'sv3_readout.md').read_text())
probe = REPORTS / 'probe_sv3'
if not (probe / 'sv2_probe_adam.md').exists():
    run([sys.executable, '-m', 'core.tools.sv2_probe', 'adam', '--run-dir', RUN_DIR[SV3], '--out-dir', LOCAL / 'probe_sv3'])
    publish(LOCAL / 'probe_sv3', probe)
print((probe / 'sv2_probe_adam.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
drive.mount('/content/drive')
print('Drive flushed -- paste REPORTS/sv3_readout.md and REPORTS/probe_sv3/sv2_probe_adam.md to Claude')
